In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================================
# 0. Path configuration
# ============================================================

def find_project_root():
    """
    Locate the project root from either a Python script or a notebook.

    Expected project structure:
    Project_Root/
    ├─ Code/
    ├─ Data/
    │  └─ SCB-Mesozoic-Granite.xls
    └─ Result/
    """
    if "__file__" in globals():
        start_dir = os.path.abspath(os.path.dirname(__file__))
    else:
        start_dir = os.path.abspath(os.getcwd())

    current = start_dir

    while True:
        if (
            os.path.exists(os.path.join(current, "Code"))
            and os.path.exists(os.path.join(current, "Data"))
            and os.path.exists(os.path.join(current, "Result"))
        ):
            return current

        parent = os.path.dirname(current)

        if parent == current:
            break

        current = parent

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code, Data, and Result folders."
    )


PROJECT_ROOT = find_project_root()

RESULT_BASE_DIR = os.path.join(PROJECT_ROOT, "Result")

DATA_ROOT = os.path.join(
    RESULT_BASE_DIR,
    "01_Foldwise_Preprocessing_and_Ratio_Features"
)

STABILITY_SUMMARY_FILE = os.path.join(
    RESULT_BASE_DIR,
    "05_Stable_Cluster_Champions_and_Novel_Ratio_Candidates",
    "rho090_stable_champions_and_ratio_candidates_summary.xlsx"
)

OUT_DIR = os.path.join(
    RESULT_BASE_DIR,
    "09_Class_Weight_Sensitivity_Analysis"
)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_XLSX = os.path.join(
    OUT_DIR,
    "09_class_weight_sensitivity_results.xlsx"
)

TYPE_COL = "Type"

IMPUTATION_METHODS = ["global_mean", "knn"]

N_OUTER_FOLDS = 5

CLASS_ORDER = ["A", "S", "I"]

MODEL_ORDER = [
    "ExtraTrees",
    "RandomForest",
    "SVM",
    "LogisticRegression"
]

CLASS_WEIGHT_SETTINGS = [
    ("None", None),
    ("balanced", "balanced")
]

FEATURE_SET_ORDER = [
    "Classical_only",
    "Classical_plus_stable_novel",
    "Stable_features_for_paper"
]

TOP_N_STABLE_RATIOS = 20

RANDOM_STATE = 42

# ============================================================
# 1. Utility functions
# ============================================================

def normalize_type_value(v):
    s = str(v).strip()

    if s in ["A", "A-type", "A-Type", "A_TYPE", "A type"]:
        return "A"
    if s in ["S", "S-type", "S-Type", "S_TYPE", "S type"]:
        return "S"
    if s in ["I", "I-type", "I-Type", "I_TYPE", "I type"]:
        return "I"

    return s


def display_feature_name(name):
    s = str(name)
    s = s.replace("R_Major_", "")
    s = s.replace("R_Trace_", "")
    s = s.replace("A12O3", "Al2O3")
    s = s.replace("10000*Ga/Al", "10000xGa/Al")
    s = s.replace("10000*Ga/A1", "10000xGa/Al")
    return s


def feature_key(name):
    s = str(name).strip()
    s = display_feature_name(s)
    s = s.replace("×", "*")
    s = s.replace("：", ":")
    s = re.sub(r"\s+", "", s)
    return s.lower()


def resolve_one_feature(feature, columns):
    columns = [str(c).strip() for c in columns]

    if feature in columns:
        return feature

    target_key = feature_key(feature)

    for c in columns:
        if feature_key(c) == target_key:
            return c

    fixed_1 = str(feature).replace("A12O3", "Al2O3")
    fixed_2 = str(feature).replace("Al2O3", "A12O3")

    for fixed in [fixed_1, fixed_2]:
        if fixed in columns:
            return fixed

        fixed_key = feature_key(fixed)

        for c in columns:
            if feature_key(c) == fixed_key:
                return c

    return None


def resolve_feature_list(features, columns):
    resolved = []
    missing = []

    for f in features:
        c = resolve_one_feature(f, columns)

        if c is None:
            missing.append(f)
        else:
            if c not in resolved:
                resolved.append(c)

    return resolved, missing


def get_classical_features(columns):
    """
    Classical geochemical-feature baseline.

    If a classical feature is not available in the current fold,
    it is skipped and a warning is printed.
    """
    aliases = {
        "A/CNK": [
            "A/CNK", "ACNK", "A_CNK"
        ],
        "A/NK": [
            "A/NK", "ANK", "A_NK"
        ],
        "10000xGa/Al": [
            "10000*Ga/Al",
            "10000xGa/Al",
            "10000×Ga/Al",
            "10000*Ga/A1",
            "Ga/Al*10000"
        ],
        "Zr+Nb+Ce+Y": [
            "Zr+Nb+Ce+Y",
            "Zr_Nb_Ce_Y",
            "Zr+Nb+Ce+Y(ppm)"
        ],
        "Sr/Y": [
            "Sr/Y",
            "R_Trace_Sr/Y"
        ],
        "Rb/Sr": [
            "Rb/Sr",
            "R_Trace_Rb/Sr"
        ],
        "K2O/Na2O": [
            "K2O/Na2O",
            "R_Major_K2O/Na2O",
            "K2O(wt%)/Na2O(wt%)",
            "R_Major_K2O(wt%)/Na2O(wt%)"
        ],
        "Fe2O3t/MgO": [
            "Fe2O3t/MgO",
            "R_Major_Fe2O3t/MgO"
        ],
    }

    selected = []

    for display_name, opts in aliases.items():
        found = None

        for f in opts:
            c = resolve_one_feature(f, columns)

            if c is not None:
                found = c
                break

        if found is None:
            print(f"Warning: classical feature was not found: {display_name}")
        else:
            selected.append(found)

    selected = list(dict.fromkeys(selected))

    return selected


def load_stable_novel_ratios():
    if not os.path.exists(STABILITY_SUMMARY_FILE):
        raise FileNotFoundError(
            f"Script 05 summary file was not found: {STABILITY_SUMMARY_FILE}"
        )

    xls = pd.ExcelFile(STABILITY_SUMMARY_FILE)

    if "stable_ratio_candidates" not in xls.sheet_names:
        raise ValueError(
            "The Script 05 summary file does not contain the "
            "'stable_ratio_candidates' sheet."
        )

    df = pd.read_excel(
        STABILITY_SUMMARY_FILE,
        sheet_name="stable_ratio_candidates"
    )

    if "Feature" not in df.columns:
        raise ValueError(
            "The stable_ratio_candidates sheet does not contain a Feature column."
        )

    if "Appearance_count" in df.columns:
        df = df[df["Appearance_count"] >= 6].copy()

    sort_cols = []
    ascending = []

    for c in [
        "Appearance_count",
        "Mean_champion_score",
        "Mean_SHAP_importance",
        "Mean_TopK_frequency"
    ]:
        if c in df.columns:
            sort_cols.append(c)
            ascending.append(False)

    if sort_cols:
        df = df.sort_values(sort_cols, ascending=ascending)

    stable_ratios = (
        df["Feature"]
        .dropna()
        .astype(str)
        .head(TOP_N_STABLE_RATIOS)
        .tolist()
    )

    return stable_ratios


def load_stable_features_for_paper():
    if not os.path.exists(STABILITY_SUMMARY_FILE):
        raise FileNotFoundError(
            f"Script 05 summary file was not found: {STABILITY_SUMMARY_FILE}"
        )

    xls = pd.ExcelFile(STABILITY_SUMMARY_FILE)

    for sheet in [
        "stable_features_for_paper",
        "core_stable_features",
        "champion_stability_summary"
    ]:
        if sheet in xls.sheet_names:
            df = pd.read_excel(STABILITY_SUMMARY_FILE, sheet_name=sheet)

            if "Feature" in df.columns:
                features = (
                    df["Feature"]
                    .dropna()
                    .astype(str)
                    .tolist()
                )

                return features

    return []


def read_fold_data(method, fold):
    method_dir = os.path.join(DATA_ROOT, method)

    train_path = os.path.join(
        method_dir,
        f"fold_{fold:02d}_train_with_ratios.xlsx"
    )

    test_path = os.path.join(
        method_dir,
        f"fold_{fold:02d}_test_with_ratios.xlsx"
    )

    if not os.path.exists(train_path):
        raise FileNotFoundError(train_path)

    if not os.path.exists(test_path):
        raise FileNotFoundError(test_path)

    train_df = pd.read_excel(train_path)
    test_df = pd.read_excel(test_path)

    train_df.columns = [str(c).strip() for c in train_df.columns]
    test_df.columns = [str(c).strip() for c in test_df.columns]

    if TYPE_COL not in train_df.columns:
        raise ValueError(f"The training data has no label column: {TYPE_COL}")

    if TYPE_COL not in test_df.columns:
        raise ValueError(f"The test data has no label column: {TYPE_COL}")

    train_df[TYPE_COL] = train_df[TYPE_COL].apply(normalize_type_value)
    test_df[TYPE_COL] = test_df[TYPE_COL].apply(normalize_type_value)

    return train_df, test_df


def clean_X(df, features):
    X = df[features].copy()

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X = X.replace([np.inf, -np.inf], np.nan)

    return X


def remove_bad_features(X_train, X_test):
    """
    Remove all-NaN, constant, or otherwise invalid columns.

    The decision is made only from the training fold.
    The test fold is only transformed by using the retained columns.
    """
    keep_cols = []

    for c in X_train.columns:
        x = pd.to_numeric(X_train[c], errors="coerce")
        x = x.replace([np.inf, -np.inf], np.nan)

        non_na = x.dropna()

        if len(non_na) == 0:
            continue

        if non_na.nunique() <= 1:
            continue

        keep_cols.append(c)

    X_train_new = X_train[keep_cols].copy()
    X_test_new = X_test[keep_cols].copy()

    return X_train_new, X_test_new, keep_cols


def build_feature_sets(train_columns):
    stable_ratios = load_stable_novel_ratios()
    stable_features = load_stable_features_for_paper()

    classical = get_classical_features(train_columns)

    novel, novel_missing = resolve_feature_list(
        stable_ratios,
        train_columns
    )

    stable_paper, stable_missing = resolve_feature_list(
        stable_features,
        train_columns
    )

    feature_sets = {
        "Classical_only": classical,
        "Classical_plus_stable_novel": list(dict.fromkeys(classical + novel)),
        "Stable_features_for_paper": stable_paper,
    }

    feature_sets = {
        k: v
        for k, v in feature_sets.items()
        if len(v) > 0
    }

    return feature_sets


def make_model(model_name, class_weight):
    if model_name == "ExtraTrees":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=600,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight=class_weight
            ))
        ])

    if model_name == "RandomForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=600,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight=class_weight
            ))
        ])

    if model_name == "SVM":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                C=10.0,
                kernel="rbf",
                gamma="scale",
                probability=False,
                random_state=RANDOM_STATE,
                class_weight=class_weight
            ))
        ])

    if model_name == "LogisticRegression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=5000,
                solver="lbfgs",
                multi_class="auto",
                C=1.0,
                random_state=RANDOM_STATE,
                class_weight=class_weight
            ))
        ])

    raise ValueError(f"Unknown model: {model_name}")


def evaluate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "Accuracy": acc,
        "Balanced_accuracy": bacc,
        "Macro_precision": macro_p,
        "Macro_recall": macro_r,
        "Macro_F1": macro_f1,
        "Weighted_precision": weighted_p,
        "Weighted_recall": weighted_r,
        "Weighted_F1": weighted_f1,
    }


def flatten_columns(df):
    df = df.copy()

    df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in df.columns
    ]

    return df


def sort_by_order(df):
    df = df.copy()

    if "Model" in df.columns:
        df["Model"] = pd.Categorical(
            df["Model"],
            categories=MODEL_ORDER,
            ordered=True
        )

    if "Imputation_method" in df.columns:
        df["Imputation_method"] = pd.Categorical(
            df["Imputation_method"],
            categories=IMPUTATION_METHODS,
            ordered=True
        )

    if "Feature_set" in df.columns:
        df["Feature_set"] = pd.Categorical(
            df["Feature_set"],
            categories=FEATURE_SET_ORDER,
            ordered=True
        )

    if "Class_weight" in df.columns:
        df["Class_weight"] = pd.Categorical(
            df["Class_weight"],
            categories=["None", "balanced"],
            ordered=True
        )

    sort_cols = [
        c for c in [
            "Model",
            "Imputation_method",
            "Feature_set",
            "Class_weight",
            "Outer_fold"
        ]
        if c in df.columns
    ]

    if sort_cols:
        df = df.sort_values(sort_cols)

    for c in ["Model", "Imputation_method", "Feature_set", "Class_weight"]:
        if c in df.columns:
            df[c] = df[c].astype(str)

    return df.reset_index(drop=True)


def save_excel_with_autowidth(path, sheets):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet_name, df in sheets.items():
            if df is None or df.empty:
                continue

            safe_sheet = sheet_name[:31]

            df.to_excel(writer, sheet_name=safe_sheet, index=False)

            ws = writer.book[safe_sheet]
            ws.freeze_panes = "A2"

            for col_cells in ws.columns:
                max_len = 0
                col_letter = col_cells[0].column_letter

                for cell in col_cells:
                    try:
                        value = str(cell.value) if cell.value is not None else ""
                        max_len = max(max_len, len(value))
                    except Exception:
                        pass

                ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 45)


# ============================================================
# 2. Main experiment: class_weight=None vs balanced
# ============================================================

all_metric_rows = []
all_classwise_rows = []
all_cm_rows = []
feature_log_rows = []

for method in IMPUTATION_METHODS:
    print(f"\n\n==================== Imputation method: {method} ====================")

    for fold in range(1, N_OUTER_FOLDS + 1):
        print(f"\n========== Outer fold {fold} ==========")

        train_df, test_df = read_fold_data(method, fold)

        feature_sets = build_feature_sets(train_df.columns)

        print("Training-set class distribution:")
        print(train_df[TYPE_COL].value_counts())

        print("Test-set class distribution:")
        print(test_df[TYPE_COL].value_counts())

        for fs_name, features in feature_sets.items():
            print(f"\n--- Feature set: {fs_name} | original feature count={len(features)} ---")

            X_train_raw = clean_X(train_df, features)
            X_test_raw = clean_X(test_df, features)

            X_train, X_test, kept_features = remove_bad_features(
                X_train_raw,
                X_test_raw
            )

            y_train = train_df[TYPE_COL].astype(str)
            y_test = test_df[TYPE_COL].astype(str)

            print(f"Feature count after cleaning={len(kept_features)}")
            print("First 10 kept features:", kept_features[:10])

            feature_log_rows.append({
                "Imputation_method": method,
                "Outer_fold": fold,
                "Feature_set": fs_name,
                "Original_n_features": len(features),
                "Kept_n_features": len(kept_features),
                "Features": "; ".join(kept_features),
            })

            for model_name in MODEL_ORDER:
                for cw_label, cw_value in CLASS_WEIGHT_SETTINGS:
                    print(f"  Training model: {model_name} | class_weight={cw_label}")

                    model = make_model(model_name, cw_value)

                    model.fit(X_train, y_train)

                    y_pred = model.predict(X_test)

                    metric_row = {
                        "Model": model_name,
                        "Imputation_method": method,
                        "Outer_fold": fold,
                        "Feature_set": fs_name,
                        "Class_weight": cw_label,
                        "N_features": len(kept_features),
                        "N_train": len(X_train),
                        "N_test": len(X_test),
                        "Features": "; ".join(kept_features),
                    }

                    metric_row.update(evaluate_metrics(y_test, y_pred))

                    all_metric_rows.append(metric_row)

                    report = classification_report(
                        y_test,
                        y_pred,
                        labels=CLASS_ORDER,
                        output_dict=True,
                        zero_division=0
                    )

                    for cls in CLASS_ORDER:
                        all_classwise_rows.append({
                            "Model": model_name,
                            "Imputation_method": method,
                            "Outer_fold": fold,
                            "Feature_set": fs_name,
                            "Class_weight": cw_label,
                            "Class": cls,
                            "Precision": report[cls]["precision"],
                            "Recall": report[cls]["recall"],
                            "F1": report[cls]["f1-score"],
                            "Support": report[cls]["support"],
                        })

                    cm = confusion_matrix(
                        y_test,
                        y_pred,
                        labels=CLASS_ORDER
                    )

                    for i, true_cls in enumerate(CLASS_ORDER):
                        for j, pred_cls in enumerate(CLASS_ORDER):
                            all_cm_rows.append({
                                "Model": model_name,
                                "Imputation_method": method,
                                "Outer_fold": fold,
                                "Feature_set": fs_name,
                                "Class_weight": cw_label,
                                "True_class": true_cls,
                                "Pred_class": pred_cls,
                                "Count": int(cm[i, j]),
                            })

                    print(
                        f"    Completed: Macro-F1={metric_row['Macro_F1']:.4f}, "
                        f"Balanced accuracy={metric_row['Balanced_accuracy']:.4f}, "
                        f"Accuracy={metric_row['Accuracy']:.4f}"
                    )


fold_metrics_df = pd.DataFrame(all_metric_rows)
classwise_df = pd.DataFrame(all_classwise_rows)
confusion_df = pd.DataFrame(all_cm_rows)
feature_log_df = pd.DataFrame(feature_log_rows)

fold_metrics_df = sort_by_order(fold_metrics_df)
classwise_df = sort_by_order(classwise_df)
confusion_df = sort_by_order(confusion_df)
feature_log_df = sort_by_order(feature_log_df)

print("\n========== Script 09 experiment completed. Preview of fold_metrics ==========")
print(fold_metrics_df.head().to_string(index=False))


# ============================================================
# 3. Summary statistics
# ============================================================

metric_cols = [
    "Accuracy",
    "Balanced_accuracy",
    "Macro_precision",
    "Macro_recall",
    "Macro_F1",
    "Weighted_precision",
    "Weighted_recall",
    "Weighted_F1"
]

summary_df = (
    fold_metrics_df
    .groupby(
        [
            "Model",
            "Imputation_method",
            "Feature_set",
            "Class_weight"
        ],
        as_index=False
    )
    .agg({
        "N_features": ["mean", "std"],
        "Accuracy": ["mean", "std"],
        "Balanced_accuracy": ["mean", "std"],
        "Macro_precision": ["mean", "std"],
        "Macro_recall": ["mean", "std"],
        "Macro_F1": ["mean", "std"],
        "Weighted_precision": ["mean", "std"],
        "Weighted_recall": ["mean", "std"],
        "Weighted_F1": ["mean", "std"],
    })
)

summary_df = flatten_columns(summary_df)

summary_df = summary_df.rename(columns={
    "Model_": "Model",
    "Imputation_method_": "Imputation_method",
    "Feature_set_": "Feature_set",
    "Class_weight_": "Class_weight"
})

summary_df = sort_by_order(summary_df)


classwise_summary_df = (
    classwise_df
    .groupby(
        [
            "Model",
            "Imputation_method",
            "Feature_set",
            "Class_weight",
            "Class"
        ],
        as_index=False
    )
    .agg({
        "Precision": ["mean", "std"],
        "Recall": ["mean", "std"],
        "F1": ["mean", "std"],
        "Support": ["mean", "std"],
    })
)

classwise_summary_df = flatten_columns(classwise_summary_df)

classwise_summary_df = classwise_summary_df.rename(columns={
    "Model_": "Model",
    "Imputation_method_": "Imputation_method",
    "Feature_set_": "Feature_set",
    "Class_weight_": "Class_weight",
    "Class_": "Class"
})

classwise_summary_df = sort_by_order(classwise_summary_df)

s_type_summary_df = classwise_summary_df[
    classwise_summary_df["Class"] == "S"
].copy()


# ============================================================
# 4. Calculate changes from class_weight=None to balanced
# ============================================================

baseline_none = summary_df[
    summary_df["Class_weight"] == "None"
].copy()

baseline_none = baseline_none[[
    "Model",
    "Imputation_method",
    "Feature_set",
    "Accuracy_mean",
    "Balanced_accuracy_mean",
    "Macro_F1_mean",
    "Macro_recall_mean",
    "Macro_precision_mean",
    "Weighted_F1_mean"
]].rename(columns={
    "Accuracy_mean": "None_Accuracy_mean",
    "Balanced_accuracy_mean": "None_Balanced_accuracy_mean",
    "Macro_F1_mean": "None_Macro_F1_mean",
    "Macro_recall_mean": "None_Macro_recall_mean",
    "Macro_precision_mean": "None_Macro_precision_mean",
    "Weighted_F1_mean": "None_Weighted_F1_mean",
})

balanced_summary = summary_df[
    summary_df["Class_weight"] == "balanced"
].copy()

delta_weight_df = balanced_summary.merge(
    baseline_none,
    on=["Model", "Imputation_method", "Feature_set"],
    how="left"
)

delta_weight_df["Delta_Accuracy_balanced_minus_None"] = (
    delta_weight_df["Accuracy_mean"] - delta_weight_df["None_Accuracy_mean"]
)

delta_weight_df["Delta_Balanced_accuracy_balanced_minus_None"] = (
    delta_weight_df["Balanced_accuracy_mean"] - delta_weight_df["None_Balanced_accuracy_mean"]
)

delta_weight_df["Delta_Macro_F1_balanced_minus_None"] = (
    delta_weight_df["Macro_F1_mean"] - delta_weight_df["None_Macro_F1_mean"]
)

delta_weight_df["Delta_Macro_recall_balanced_minus_None"] = (
    delta_weight_df["Macro_recall_mean"] - delta_weight_df["None_Macro_recall_mean"]
)

delta_weight_df["Delta_Macro_precision_balanced_minus_None"] = (
    delta_weight_df["Macro_precision_mean"] - delta_weight_df["None_Macro_precision_mean"]
)

delta_weight_df["Delta_Weighted_F1_balanced_minus_None"] = (
    delta_weight_df["Weighted_F1_mean"] - delta_weight_df["None_Weighted_F1_mean"]
)

delta_weight_df = sort_by_order(delta_weight_df)


# S-type delta
s_none = s_type_summary_df[
    s_type_summary_df["Class_weight"] == "None"
].copy()

s_none = s_none[[
    "Model",
    "Imputation_method",
    "Feature_set",
    "Precision_mean",
    "Recall_mean",
    "F1_mean"
]].rename(columns={
    "Precision_mean": "None_S_precision_mean",
    "Recall_mean": "None_S_recall_mean",
    "F1_mean": "None_S_F1_mean"
})

s_balanced = s_type_summary_df[
    s_type_summary_df["Class_weight"] == "balanced"
].copy()

s_delta_df = s_balanced.merge(
    s_none,
    on=["Model", "Imputation_method", "Feature_set"],
    how="left"
)

s_delta_df["Delta_S_precision_balanced_minus_None"] = (
    s_delta_df["Precision_mean"] - s_delta_df["None_S_precision_mean"]
)

s_delta_df["Delta_S_recall_balanced_minus_None"] = (
    s_delta_df["Recall_mean"] - s_delta_df["None_S_recall_mean"]
)

s_delta_df["Delta_S_F1_balanced_minus_None"] = (
    s_delta_df["F1_mean"] - s_delta_df["None_S_F1_mean"]
)

s_delta_df = sort_by_order(s_delta_df)


# Win-rate statistics
weight_win_summary_df = (
    delta_weight_df
    .groupby(["Feature_set"], as_index=False)
    .agg(
        N_comparisons=("Delta_Macro_F1_balanced_minus_None", "count"),
        N_Macro_F1_improved=(
            "Delta_Macro_F1_balanced_minus_None",
            lambda x: int((x > 0).sum())
        ),
        Mean_delta_Macro_F1=("Delta_Macro_F1_balanced_minus_None", "mean"),
        Median_delta_Macro_F1=("Delta_Macro_F1_balanced_minus_None", "median"),
        N_Balanced_accuracy_improved=(
            "Delta_Balanced_accuracy_balanced_minus_None",
            lambda x: int((x > 0).sum())
        ),
        Mean_delta_Balanced_accuracy=("Delta_Balanced_accuracy_balanced_minus_None", "mean"),
        Median_delta_Balanced_accuracy=("Delta_Balanced_accuracy_balanced_minus_None", "median"),
    )
)

weight_win_summary_df["Positive_rate_Macro_F1"] = (
    weight_win_summary_df["N_Macro_F1_improved"]
    / weight_win_summary_df["N_comparisons"]
)

weight_win_summary_df["Positive_rate_Balanced_accuracy"] = (
    weight_win_summary_df["N_Balanced_accuracy_improved"]
    / weight_win_summary_df["N_comparisons"]
)


s_weight_win_summary_df = (
    s_delta_df
    .groupby(["Feature_set"], as_index=False)
    .agg(
        N_comparisons=("Delta_S_recall_balanced_minus_None", "count"),
        N_S_recall_improved=(
            "Delta_S_recall_balanced_minus_None",
            lambda x: int((x > 0).sum())
        ),
        Mean_delta_S_recall=("Delta_S_recall_balanced_minus_None", "mean"),
        Median_delta_S_recall=("Delta_S_recall_balanced_minus_None", "median"),
        N_S_F1_improved=(
            "Delta_S_F1_balanced_minus_None",
            lambda x: int((x > 0).sum())
        ),
        Mean_delta_S_F1=("Delta_S_F1_balanced_minus_None", "mean"),
        Median_delta_S_F1=("Delta_S_F1_balanced_minus_None", "median"),
    )
)

s_weight_win_summary_df["Positive_rate_S_recall"] = (
    s_weight_win_summary_df["N_S_recall_improved"]
    / s_weight_win_summary_df["N_comparisons"]
)

s_weight_win_summary_df["Positive_rate_S_F1"] = (
    s_weight_win_summary_df["N_S_F1_improved"]
    / s_weight_win_summary_df["N_comparisons"]
)


# ============================================================
# 5. Figures
# ============================================================

def save_grouped_bar_for_weight(summary_df, feature_set, metric, out_png, title):
    sub = summary_df[
        summary_df["Feature_set"] == feature_set
    ].copy()

    if sub.empty:
        return

    metric_col = f"{metric}_mean"

    if metric_col not in sub.columns:
        return

    sub["Label"] = (
        sub["Model"].astype(str)
        + " | "
        + sub["Imputation_method"].astype(str)
        + " | "
        + sub["Class_weight"].astype(str)
    )

    sub = sub.sort_values(
        ["Model", "Imputation_method", "Class_weight"]
    )

    plt.figure(figsize=(12, max(5, 0.32 * len(sub))), dpi=300)

    plt.barh(sub["Label"], sub[metric_col])

    plt.xlabel(metric.replace("_", " "), fontsize=12, fontweight="bold")
    plt.ylabel("Model | Imputation | class_weight", fontsize=12, fontweight="bold")
    plt.title(title, fontsize=14, fontweight="bold")

    for i, v in enumerate(sub[metric_col].values):
        plt.text(v, i, f"{v:.3f}", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


def save_delta_bar(delta_df, delta_col, out_png, title):
    if delta_df.empty or delta_col not in delta_df.columns:
        return

    sub = delta_df.copy()

    sub["Label"] = (
        sub["Model"].astype(str)
        + " | "
        + sub["Imputation_method"].astype(str)
        + " | "
        + sub["Feature_set"].astype(str)
    )

    sub = sub.sort_values(delta_col, ascending=True)

    plt.figure(figsize=(11, max(6, 0.28 * len(sub))), dpi=300)

    plt.barh(sub["Label"], sub[delta_col])
    plt.axvline(0, linestyle="--", linewidth=1)

    plt.xlabel(delta_col, fontsize=12, fontweight="bold")
    plt.ylabel("Model | Imputation | Feature set", fontsize=12, fontweight="bold")
    plt.title(title, fontsize=14, fontweight="bold")

    for i, v in enumerate(sub[delta_col].values):
        plt.text(v, i, f"{v:.3f}", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


def save_heatmap(pivot_df, out_png, title, value_label):
    if pivot_df.empty:
        return

    row_labels = pivot_df.index.astype(str).tolist()
    col_labels = pivot_df.columns.astype(str).tolist()
    values = pivot_df.values.astype(float)

    plt.figure(
        figsize=(
            max(7, 1.5 * len(col_labels)),
            max(5, 0.38 * len(row_labels))
        ),
        dpi=300
    )

    im = plt.imshow(values, aspect="auto")
    plt.colorbar(im, label=value_label)

    plt.xticks(
        np.arange(len(col_labels)),
        col_labels,
        rotation=35,
        ha="right",
        fontsize=10
    )

    plt.yticks(
        np.arange(len(row_labels)),
        row_labels,
        fontsize=9
    )

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            val = values[i, j]

            if not np.isnan(val):
                plt.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8
                )

    plt.title(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


# Figure 1: Macro-F1 comparison for Stable_features_for_paper
save_grouped_bar_for_weight(
    summary_df,
    "Stable_features_for_paper",
    "Macro_F1",
    os.path.join(OUT_DIR, "09_macroF1_class_weight_Stable_features_for_paper.png"),
    "Macro-F1 under class_weight settings: Stable features for paper"
)

# Figure 2: Balanced accuracy comparison for Stable_features_for_paper
save_grouped_bar_for_weight(
    summary_df,
    "Stable_features_for_paper",
    "Balanced_accuracy",
    os.path.join(OUT_DIR, "09_balanced_accuracy_class_weight_Stable_features_for_paper.png"),
    "Balanced accuracy under class_weight settings: Stable features for paper"
)

# Figure 3: Macro-F1 difference from balanced to None
save_delta_bar(
    delta_weight_df,
    "Delta_Macro_F1_balanced_minus_None",
    os.path.join(OUT_DIR, "09_delta_macroF1_balanced_minus_None.png"),
    "Delta Macro-F1: class_weight=balanced minus None"
)

# Figure 4: S-type recall difference from balanced to None
save_delta_bar(
    s_delta_df,
    "Delta_S_recall_balanced_minus_None",
    os.path.join(OUT_DIR, "09_delta_S_recall_balanced_minus_None.png"),
    "Delta S-type recall: class_weight=balanced minus None"
)

# Figure 5: S-type F1 difference from balanced to None
save_delta_bar(
    s_delta_df,
    "Delta_S_F1_balanced_minus_None",
    os.path.join(OUT_DIR, "09_delta_S_F1_balanced_minus_None.png"),
    "Delta S-type F1: class_weight=balanced minus None"
)

# Figure 6: S-type F1 heatmap
s_type_summary_df["Model_method_weight"] = (
    s_type_summary_df["Model"].astype(str)
    + " | "
    + s_type_summary_df["Imputation_method"].astype(str)
    + " | "
    + s_type_summary_df["Class_weight"].astype(str)
)

s_f1_pivot = s_type_summary_df.pivot_table(
    index="Model_method_weight",
    columns="Feature_set",
    values="F1_mean",
    aggfunc="first"
)

s_f1_pivot = s_f1_pivot[
    [c for c in FEATURE_SET_ORDER if c in s_f1_pivot.columns]
]

save_heatmap(
    s_f1_pivot,
    os.path.join(OUT_DIR, "09_S_type_F1_heatmap.png"),
    "S-type F1 under class-weight settings",
    "Mean S-type F1"
)

# Figure 7: S-type recall heatmap
s_recall_pivot = s_type_summary_df.pivot_table(
    index="Model_method_weight",
    columns="Feature_set",
    values="Recall_mean",
    aggfunc="first"
)

s_recall_pivot = s_recall_pivot[
    [c for c in FEATURE_SET_ORDER if c in s_recall_pivot.columns]
]

save_heatmap(
    s_recall_pivot,
    os.path.join(OUT_DIR, "09_S_type_recall_heatmap.png"),
    "S-type recall under class-weight settings",
    "Mean S-type recall"
)


# ============================================================
# 6. Save Excel results
# ============================================================

sheets = {
    "fold_metrics": fold_metrics_df,
    "summary_by_weight": summary_df,
    "classwise_metrics": classwise_df,
    "classwise_summary": classwise_summary_df,
    "S_type_summary": s_type_summary_df.drop(columns=["Model_method_weight"], errors="ignore"),
    "delta_balanced_vs_None": delta_weight_df,
    "S_type_delta": s_delta_df,
    "weight_win_summary": weight_win_summary_df,
    "S_weight_win_summary": s_weight_win_summary_df,
    "confusion_matrix_long": confusion_df,
    "feature_log": feature_log_df,
}

save_excel_with_autowidth(OUT_XLSX, sheets)


# ============================================================
# 7. Print core results
# ============================================================

print("\n========== Script 09 class-weight sensitivity analysis completed ==========")
print("Output directory:", OUT_DIR)
print("Excel file:", OUT_XLSX)

print("\n========== Preview of summary_by_weight ==========")
print(summary_df.head(20).to_string(index=False))

print("\n========== Overall metric changes: balanced minus None ==========")
show_cols = [
    "Model",
    "Imputation_method",
    "Feature_set",
    "Delta_Accuracy_balanced_minus_None",
    "Delta_Balanced_accuracy_balanced_minus_None",
    "Delta_Macro_F1_balanced_minus_None",
    "Delta_Macro_recall_balanced_minus_None",
]
show_cols = [c for c in show_cols if c in delta_weight_df.columns]

print(delta_weight_df[show_cols].reset_index(drop=True).to_string(index=False))

print("\n========== S-type metric changes: balanced minus None ==========")
show_cols_s = [
    "Model",
    "Imputation_method",
    "Feature_set",
    "Delta_S_precision_balanced_minus_None",
    "Delta_S_recall_balanced_minus_None",
    "Delta_S_F1_balanced_minus_None",
]
show_cols_s = [c for c in show_cols_s if c in s_delta_df.columns]

print(s_delta_df[show_cols_s].reset_index(drop=True).to_string(index=False))

print("\n========== Win rate of class_weight for overall Macro-F1 / Balanced accuracy ==========")
print(weight_win_summary_df.to_string(index=False))

print("\n========== Win rate of class_weight for S-type Recall / F1 ==========")
print(s_weight_win_summary_df.to_string(index=False))